In [35]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import xml.etree.ElementTree as ET
import numpy as np
from pathlib import Path
import os
import re

Načítání dat

Pro správné fungování změňte proměnnou „path_to_documents“, která by měla obsahovat cestu k trénovací sadě dokumentů.

In [ ]:
CONTRACTIONS = {
    "it's": "it is",
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "i'm": "i am",
    "you're": "you are",
    "he's": "he is",
    "she's": "she is",
    "we're": "we are",
    "they're": "they are",
    "can't": "cannot",
    "won't": "will not",
    "n't": " not", 
    "'re": " are",
    "'s": " is",
    "'d": " would",
    "'ll": " will",
    "'ve": " have"
}



def expand_contractions(text):
    text = text.lower()

    for contraction, full in CONTRACTIONS.items():
        text = re.sub(rf"\b{re.escape(contraction)}\b", full, text)

    return text




def is_numeric_row(line: str) -> bool:
    return bool(re.fullmatch(r'[\d\s\t]+', line.strip()))



def remove_garbage(text: str) -> str:
    lines = text.strip().splitlines()

    lines = [line for line in lines if line.strip()]

    #remove tags <html> and <pre>
    lines = lines[2:-2]

    
    i = len(lines) - 1
    
    while i >= 0 and is_numeric_row(lines[i]):
        i -= 1
    
    cleaned_lines = lines[:i+1]

    
    return " ".join(cleaned_lines)


path_to_documents = Path(r"C:\Users\Yauheni\studium\data_heap\SZPJ_SP1_collection\documents")

raw_data = {}


for file in path_to_documents.glob("*.html"):
    content = file.read_text(encoding="utf-8")
    bname = os.path.basename(file).split(".")[0]

    raw_data[bname] = remove_garbage(content)

Předzpracování dat

Všechna slova byla převedena na malá písmena.
Speciální znaky byly nahrazeny mezerou: „.“, „?“, „,“, „!“, „;“, „-“, „:“.

Byl také vyzkoušen převod kontrakcí, například nahrazení „it's“ na „it is“. Tento postup však vedl k horším výsledkům než předchozí, proto byl odstraněn.

In [ ]:


for key, text in raw_data.items():
    lowercase_text = text.lower()

    #filterd = expand_contractions(lowercase_text)
    
    filterd = lowercase_text.replace(",", " ")
    filterd = filterd.replace(".", " ")
    filterd = filterd.replace(":", " ")
    filterd = filterd.replace("!", " ")
    filterd = filterd.replace("?", " ")
    filterd = filterd.replace(";", " ")
    filterd = filterd.replace("-", " ")

    filterd = re.sub(r' {2,}', ' ', filterd)

    raw_data[key] = filterd

preliminary report international algebraic language cacm december 1958 perlis a j samelson k ca581203 jb march 22 1978 8 28 pm


TF-IDF vektorizér

Pro zlepšení výkonu modelu byly použity následující režimy: sublinear_tf, smooth_idf, norm='l2'.

Byly také testovány další úpravy, ale ty vedly k horším výsledkům: ngram_range, max_df, min_df.


In [ ]:

data = []
keys = []


for key, text in raw_data.items():
    data.append(text)
    keys.append(key)


vectorizer = TfidfVectorizer(sublinear_tf=True, smooth_idf=True, norm='l2')
vec_mat = vectorizer.fit_transform(data)


(3204, 15038)


Přečíst XML soubor, najít dokumenty a zapsat výsledek do textového souboru result.txt

Pro správné fungování změňte proměnné:

    „path_to_quarys“ – cesta k XML souboru s dotazy
    „path_to_output“ – cesta k výstupnímu textovému souboru result.txt

In [ ]:

###### read XML file with quarys #######

path_to_quarys = r"C:\Users\Yauheni\studium\data_heap\SZPJ_SP1_collection\query_devel.xml"
path_to_output = r"C:\Users\Yauheni\studium\data_heap\SZPJ_SP1_collection\result_cl.txt"

with open(path_to_quarys, "r", encoding="utf-8") as f:
    content = f.read()


content = "<ROOT>" + content + "</ROOT>"

root = ET.fromstring(content)

data = {}

for doc in root.findall("DOC"):
    docno = int(doc.find("DOCNO").text.strip())

    text = " ".join(doc.itertext())

    text = text.replace(str(docno), "")
    text = re.sub(r"\s+", " ", text).strip()

    lowercase_text = text.lower()

    #filterd = expand_contractions(lowercase_text)
    
    filterd = lowercase_text.replace(",", " ")
    filterd = filterd.replace(".", " ")
    filterd = filterd.replace(":", " ")
    filterd = filterd.replace("!", " ")
    filterd = filterd.replace("?", " ")
    filterd = filterd.replace(";", " ")
    filterd = filterd.replace("-", " ")

    filterd = re.sub(r' {2,}', ' ', filterd)
    


    data[docno] = filterd




with open(path_to_output, "w", encoding="utf-8") as out:

    for q_id, query in data.items():

        q_vec = vectorizer.transform([query])

        scores = cosine_similarity(q_vec, vec_mat)[0]

        top_100 = np.argsort(scores)[::-1][:100]

        for idx in top_100:
            doc_id = keys[idx]
            score = scores[idx]

            out.write(f"{q_id}\t{doc_id}\t{score}\n")
